# Churnalism 분석 템플릿 — 사건 교체식 (고려아연·홈플러스·…)

네이버 뉴스에서 한 **사건(토픽)** 의 기사를 일자별 전수 수집해, **일별 기사수(N)** 와 **기사 간 평균 코사인유사도(S)** 가 이슈 발생 전후로 유의하게 달라지는지(=churnalism, 유사기사 대량생산)를 통계 검정합니다. 마지막에 **언론사 간 유사도**까지 봅니다.

**사용법**: `[0]`에서 `TOPIC_CODE` 한 줄만 바꾸면 다른 사건에 그대로 적용됩니다.

| 단계 | 내용 |
|---|---|
| [0] 설정 | 토픽 프리셋 선택(검색어·필터어·이슈일·기간) |
| [1] 수집 | more-API 커서로 하루 전수, 재개 가능 |
| [2] 필터 | 그 사건 관련 기사만(시황·타사 incidental 제외) |
| [3] 임베딩 | 제목 → text-embedding-3-large (또는 KoSBERT) |
| [4] 일자별 N·S | 기사수 + 평균 pairwise cosine |
| [5] 시각화 | N·S 시계열 + 평소 기준선(μ±3σ) |
| [6] 검정 | N=음이항회귀 · S=permutation+무작위 영모형 z · CUSUM (news-pushing-book) |
| [7] 임계 초과 ↔ 이슈 | 급증일 검출 + 헤드라인으로 사건 자동 식별 |
| [8] 언론사 간 유사도 | 언론사별 centroid → 언론사×언론사 코사인 히트맵 |

> 검정 방법론 출처: https://sdkparkforbi.github.io/news-pushing-book/ — N은 음이항회귀, S는 permutation + 무작위배정 영모형(z), 관리도(μ+3σ)·CUSUM. 임베딩 기본은 SBERT(ko-sroberta)이며 [3]에서 OpenAI로 교체 가능.

## [0] 설정 · 토픽 선택

In [ ]:
# ================================================================
# [0] 설정 — 토픽 프리셋에서 TOPIC_CODE 한 줄만 선택/추가
# ================================================================

import subprocess, sys
for p in ["openai","statsmodels","scikit-learn","koreanize-matplotlib","beautifulsoup4","lxml"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p], check=False)
import os, re, time, random, json as _json, csv
import numpy as np, pandas as pd, requests, urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
try: import koreanize_matplotlib
except Exception: pass
urllib3.disable_warnings()

# ── 사건(토픽) 프리셋 ─────────────────────────────────────────────
#   query : 네이버 검색어
#   incl  : 제목에 이 중 하나가 있어야 '그 사건' 기사로 인정(필터)
#   event : 이슈 발생 기준일(이 날 이전=평소/사전, 이후=사후)
#   start/end : 수집 기간(이슈일 좌우로 넉넉히)
PRESETS = {
    "고려아연": dict(query="고려아연", name="고려아연 M&A",
                  incl=["고려아연","영풍"], event="2024-09-13",
                  start="2023-09-13", end="2025-09-13"),
    "홈플러스": dict(query="홈플러스", name="홈플러스 사태",
                  incl=["홈플러스","MBK"], event="2025-03-04",   # MBK 홈플러스 기업회생 신청(※날짜 확인 후 수정)
                  start="2024-03-04", end="2025-12-31"),
}

TOPIC_CODE = "고려아연"          # ★★★ 분석할 사건만 여기서 바꾸세요 ★★★
P = PRESETS[TOPIC_CODE]
QUERY, TOPIC_NAME = P["query"], P["name"]
INCL = P["incl"]
EVENT_TIME = pd.Timestamp(P["event"])
START_DATE, END_DATE = P["start"], P["end"]
TIME_UNIT = "D"
EMBED_MODEL = "text-embedding-3-large"
N_PERM, N_NULLSIM = 5000, 1000
# 산출 파일명(토픽코드 자동 부착)
OUT_COLLECT = f"collect_{TOPIC_CODE}.csv"
OUT_NS      = f"daily_{TOPIC_CODE}_NS.csv"
PNG_MAIN    = f"churnalism_{TOPIC_CODE}.png"
PNG_SIGNAL  = f"signal_{TOPIC_CODE}.png"
PNG_MEDIA   = f"media_{TOPIC_CODE}.png"

assert os.environ.get("OPENAI_API_KEY") or True, "OpenAI 사용 시 OPENAI_API_KEY 필요(또는 [3] SBERT 사용)"
print(f"[{TOPIC_CODE}] {TOPIC_NAME} | {START_DATE}~{END_DATE} | 이슈 {EVENT_TIME.date()} | 필터어 {INCL}")

## [1] 수집 (일자별 전수)

In [ ]:
# ================================================================
# [1] 수집 — more-API 일자별 전수(최신순), 재개 가능
# ================================================================

UA="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
def mk():
    s=requests.Session(); s.headers.update({"User-Agent":UA,"Accept-Language":"ko-KR,ko;q=0.9"})
    s.mount("https://",HTTPAdapter(max_retries=Retry(total=2,backoff_factor=0.6,status_forcelist=[429,500,502,503])))
    return s
def warm(s):
    try:
        s.get("https://www.naver.com/",timeout=15); time.sleep(1.2)
        s.get("https://search.naver.com/",timeout=15,headers={"Referer":"https://www.naver.com/"}); time.sleep(1.2)
    except Exception: pass
def _norm(u):
    u=str(u).strip()
    if u.startswith("http://"): u="https://"+u[7:]
    if u.startswith("https://www."): u="https://"+u[12:]
    return u.rstrip("/")
def collect_day(s, ymd):
    d=f"{ymd[:4]}.{ymd[4:6]}.{ymd[6:]}"; q=requests.utils.quote(QUERY); out={}
    def grab(html):
        soup=BeautifulSoup(html,"lxml")
        tit=soup.select('a[nocr="1"][data-heatmap-target=".tit"]'); prs=soup.select('.sds-comps-profile-info-title-text')
        for i,t in enumerate(tit):
            h=t.get("href")
            if not h: continue
            sp=t.select_one(".sds-comps-text-type-headline1")
            out.setdefault(_norm(h),{"press":prs[i].get_text(strip=True) if i<len(prs) else "미상",
                                     "title":sp.get_text(strip=True) if sp else (t.get_text(strip=True) or "제목없음"),"url":_norm(h)})
    nso=requests.utils.quote(f"so:dd,p:from{ymd}to{ymd}")
    page=("https://search.naver.com/search.naver?where=news&query="+q+f"&sm=tab_opt&sort=1&photo=0&field=0&pd=3&ds={d}&de={d}&start=1&mynews=0&nso="+nso)
    try: html=s.get(page,timeout=30,headers={"Referer":"https://search.naver.com/"}).text
    except Exception: return []
    grab(html)
    anc="api"+chr(92)+"/tab"+chr(92)+"/more"; i=html.find(anc)
    if i>=0:
        try: more=_json.loads('"'+html[html.rfind("https:",0,i):html.find(chr(34),i)]+'"')
        except Exception: more=None
        hdr={"Referer":page,"Accept":"*/*","X-Requested-With":"XMLHttpRequest"}
        while more:
            try: j=s.get(more,timeout=30,headers=hdr).json()
            except Exception: time.sleep(2); break
            grab(j.get("collection",[{}])[0].get("html",""))
            more=j.get("url","")
            if more: time.sleep(random.uniform(0.35,0.7))
    return list(out.values())

done=set()
if os.path.exists(OUT_COLLECT):
    done=set(pd.read_csv(OUT_COLLECT,usecols=["date"])["date"].astype(str))
else:
    with open(OUT_COLLECT,"w",newline="",encoding="utf-8-sig") as f: csv.writer(f).writerow(["date","press","title","url"])
print(f"이미 수집 {len(done)}일 (재실행하면 빠진 날만 이어서)")

s=mk(); warm(s)
day=datetime.strptime(START_DATE,"%Y-%m-%d"); end=datetime.strptime(END_DATE,"%Y-%m-%d"); nd=0
while day<=end:
    ds=day.strftime("%Y-%m-%d")
    if ds in done: day+=timedelta(days=1); continue
    try: arts=collect_day(s, day.strftime("%Y%m%d"))
    except Exception: s=mk(); warm(s); arts=[]
    with open(OUT_COLLECT,"a",newline="",encoding="utf-8-sig") as f:
        w=csv.writer(f)
        for a in arts: w.writerow([ds,a["press"],a["title"],a["url"]])
    nd+=1
    if nd%20==0: print(f"  {ds}: {len(arts)}건")
    day+=timedelta(days=1); time.sleep(random.uniform(0.6,1.2))
print("수집 완료 →", OUT_COLLECT)

## [2] 필터 (그 사건 관련 기사만)

In [ ]:
# ================================================================
# [2] 필터 — 제목에 사건 키워드(INCL)가 든 기사만, 명백한 비관련 제외
# ================================================================

df = pd.read_csv(OUT_COLLECT).dropna(subset=["title"]).drop_duplicates("url")
OFF = ["봉사","따밥","날씨","한파","레시피","맛집","여행 명소","개봉","예능","부고","인사동정","오늘의 운세","띠별","운세"]
def keep(t):
    t=str(t)
    return any(k in t for k in INCL) and not any(o in t for o in OFF)
n0=len(df)
df = df[df["title"].apply(keep)].copy()
df["published_at"]=pd.to_datetime(df["date"]); df=df.sort_values("published_at").reset_index(drop=True)
print(f"필터: {n0:,} → {len(df):,}건 (제목에 {INCL} 포함) | {df.published_at.min().date()} ~ {df.published_at.max().date()}")
# 참고: 제목에 사건명이 없지만 본문상 관련인 기사는 일부 누락될 수 있음(정밀도 우선)

## [3] 임베딩 (제목)

In [ ]:
# ================================================================
# [3] 임베딩 — 제목 → 단위벡터 (기본 OpenAI, 주석 해제로 KoSBERT)
# ================================================================

from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
texts = df["title"].astype(str).tolist(); emb=[]
for i in range(0,len(texts),256):
    for _ in range(4):
        try:
            r=client.embeddings.create(model=EMBED_MODEL,input=texts[i:i+256])
            emb.extend([d.embedding for d in r.data]); break
        except Exception as e: print("재시도:",str(e)[:60]); time.sleep(3)
    if (i//256)%10==0: print(f"  {min(i+256,len(texts))}/{len(texts)}")
emb=np.asarray(emb,dtype=np.float32); emb/=np.linalg.norm(emb,axis=1,keepdims=True)
print("임베딩:", emb.shape)

# ── KoSBERT 대안(무료·로컬, news-pushing-book 기본) ──
# from sentence_transformers import SentenceTransformer
# emb = SentenceTransformer("jhgan/ko-sroberta-multitask").encode(texts, normalize_embeddings=True)

## [4] 일자별 N·S

In [ ]:
# ================================================================
# [4] 일자별 N(기사수) · S(평균 코사인유사도)
# ================================================================

def mean_pairwise_cosine(v):
    n=len(v)
    if n<2: return np.nan
    m=v.mean(axis=0); return (n*float(m@m)-1.0)/(n-1)   # 단위벡터일 때 빠른 공식
df["bucket"]=df["published_at"].dt.floor(TIME_UNIT)
rows=[]
for b,g in df.groupby("bucket"):
    idx=g.index.values
    rows.append({"bucket":b,"N":len(idx),"S":mean_pairwise_cosine(emb[idx]),
                 "period":"pre" if b<EVENT_TIME else "post"})
ts=pd.DataFrame(rows).sort_values("bucket").reset_index(drop=True)
ts.to_csv(OUT_NS,index=False,encoding="utf-8-sig")
pre=ts[ts.period=="pre"]; muN,sdN=pre.N.mean(),pre.N.std(ddof=1); muS,sdS=pre.S.mean(),pre.S.std(ddof=1)
print(f"일자 {len(ts)}개 | 사전 {len(pre)} · 사후 {len(ts)-len(pre)} | 평소 N={muN:.1f}±{sdN:.1f}, S={muS:.3f}±{sdS:.3f}")

## [5] 시각화

In [ ]:
# ================================================================
# [5] 시각화 — N·S 시계열 + 평소 기준선(μ, μ+3σ)
# ================================================================

ev=(ts.bucket<EVENT_TIME).sum()
fig,ax=plt.subplots(2,1,figsize=(15,7),sharex=True)
col=["#5BA8FF" if p=="pre" else "#FF7A6B" for p in ts.period]
ax[0].bar(range(len(ts)),ts.N,color=col,width=1.0)
ax[0].axhline(muN,ls="--",color="#E0954F",lw=1.5,label=f"평소 평균 {muN:.1f}")
ax[0].axhline(muN+3*sdN,ls=":",color="#C23A2B",lw=1.5,label="평소 +3σ")
ax[0].axvline(ev-0.5,color="gray"); ax[0].set_ylabel("기사 수 N"); ax[0].legend(fontsize=9)
ax[0].set_title(f"{TOPIC_NAME} — 일자별 기사 수 (회색선=이슈 {EVENT_TIME.date()})")
ax[1].bar(range(len(ts)),ts.S,color=col,width=1.0)
ax[1].axhline(muS,ls="--",color="#E0954F",lw=1.5,label=f"평소 평균 {muS:.3f}")
ax[1].axhline(muS+3*sdS,ls=":",color="#C23A2B",lw=1.5,label="평소 +3σ")
ax[1].axvline(ev-0.5,color="gray"); ax[1].set_ylabel("평균 코사인 S"); ax[1].legend(fontsize=9)
st=max(1,len(ts)//24); ax[1].set_xticks(range(0,len(ts),st))
ax[1].set_xticklabels([ts.bucket.iloc[i].strftime("%y-%m-%d") for i in range(0,len(ts),st)],rotation=45,ha="right",fontsize=8)
plt.tight_layout(); plt.savefig(PNG_MAIN,dpi=120); plt.show()

## [6] Churnalism 검정 (N · S · CUSUM)

In [ ]:
# ================================================================
# [6] 검정 — ①음이항회귀(N) ②permutation+영모형 z(S) ③CUSUM · 종합판정
# ================================================================

import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu

# ── ① 기사 수 N: 음이항 회귀 (요일 통제, post β1>0 단측) ──
d=ts.copy(); d["post"]=(d.period=="post").astype(int); d["dow"]=d.bucket.dt.dayofweek.astype("category")
poi=smf.glm("N ~ post + C(dow)",data=d,family=sm.families.Poisson()).fit()
aux=((d.N-poi.mu)**2-d.N)/poi.mu; alpha=max(sm.OLS(aux,poi.mu).fit().params.iloc[0],1e-6)
nbm=smf.glm("N ~ post + C(dow)",data=d,family=sm.families.NegativeBinomial(alpha=alpha)).fit()
b1=nbm.params["post"]; p1=nbm.pvalues["post"]/2 if b1>0 else 1-nbm.pvalues["post"]/2
print(f"① 기사수 N: exp(β1)={np.exp(b1):.2f}배, 단측 p={p1:.3g} → {'유의 증가' if (b1>0 and p1<0.05) else '근거 약함'}")

# ── ② 평균 유사도 S(원자료): permutation (기사<2일 NaN 제외) ──
vd=ts.dropna(subset=["S"])
obs=vd[vd.period=="post"].S.mean()-vd[vd.period=="pre"].S.mean()
Sv=vd.S.values; npost=(vd.period=="post").sum(); rng=np.random.default_rng(1)
perm=np.array([(lambda sh: sh[:npost].mean()-sh[npost:].mean())(rng.permutation(Sv)) for _ in range(N_PERM)])
p2=(np.sum(perm>=obs)+1)/(N_PERM+1)
print(f"② 유사도 S(원자료): 사후-사전={obs:+.4f}, permutation p={p2:.3g} → {'유의 증가' if (obs>0 and p2<0.05) else '평탄(평소에도 주제 유사)'}")

# ── ②-보정 무작위배정 영모형 z (기사수 효과 제거): churnalism 핵심 지표 ──
def null_sim(n,nsim,pool,rng):
    if n<2: return np.nan,np.nan
    a=np.array([mean_pairwise_cosine(pool[rng.choice(len(pool),n,replace=False)]) for _ in range(nsim)])
    return a.mean(),a.std(ddof=1)
rng=np.random.default_rng(7); zz=[]
for _,r in ts.iterrows():
    mu_,sd_=null_sim(int(r.N),N_NULLSIM,emb,rng)
    zz.append((r.S-mu_)/sd_ if (sd_ and sd_>0) else np.nan)
ts["z"]=zz
zpre=ts[ts.period=="pre"].z.dropna(); zpost=ts[ts.period=="post"].z.dropna()
u,p3=mannwhitneyu(zpost,zpre,alternative="greater")
print(f"②보정 z: 사전중앙 {zpre.median():.2f} vs 사후중앙 {zpost.median():.2f}, MWU p={p3:.3g} → {'보정 후도 유의' if p3<0.05 else '근거 약함'}")

# ── ③ CUSUM: 평소평균 대비 누적합으로 급증 시작구간 검출 ──
k=0.5*sdN; cN=np.maximum.accumulate(np.zeros(len(ts)))  # placeholder
c=np.zeros(len(ts)); acc=0.0
for i,v in enumerate(ts.N.values):
    acc=max(0.0, acc+(v-muN)-k); c[i]=acc
ts["cusumN"]=c; hN=5*sdN
first=ts[ts.cusumN>hN].bucket.min() if (ts.cusumN>hN).any() else None
print(f"③ CUSUM: 기사수 누적이탈이 한계(5σ) 초과 시작일 = {first.date() if first is not None else '없음'}")

# ── 종합 판정 : N 급증 + 무작위대비 동질성 사후강화 ──
churn=(b1>0 and p1<0.05) and (p3<0.05)
print("\n"+"="*54)
print(f"종합 판정: {'✅ CHURNALISM (기사수↑ + 무작위대비 동질성 사후 강화)' if churn else '△ 일부 조건만'}")
print(f"  · 절대유사도 S는 평소에도 높아 변화 작음(p={p2:.3g}) → 판정은 보정 z 기준")
print("="*54)

## [7] 임계 초과 시점 ↔ 중대 이슈 매칭
기사수 N이 **평소+3σ 임계**를 넘는 날을 검출하고 **그날 대표 헤드라인**을 함께 출력해, 각 급증이 어떤 사건(발표·결과·소송·압수수색 등)인지 데이터로 자동 식별합니다.

In [ ]:
# ================================================================
# [7] 임계 초과(평소+3σ) 시점 + 헤드라인으로 이슈 식별
# ================================================================

thrN=muN+3*sdN; thrS=muS+3*sdS
print(f"기사수 임계(평소 {muN:.1f}+3σ={thrN:.1f}) 초과일: {(ts.N>thrN).sum()}일 | 유사도 임계({thrS:.3f}) 초과일: {(ts.S>thrS).sum()}일")
peaks=ts[ts.N>thrN].nlargest(15,"N").sort_values("bucket")
print("\n[기사수 급증(임계 초과) 시점 ↔ 그날 대표 헤드라인]")
for _,r in peaks.iterrows():
    titles=df[df.bucket==r.bucket]["title"].tolist()
    print(f"\n● {r.bucket.date()}  N={int(r.N)} (평소의 {r.N/muN:.0f}배)  S={r.S:.3f}")
    for t in titles[:3]: print("   ·", str(t)[:54])

fig,ax=plt.subplots(figsize=(15,5))
col=["#5BA8FF" if p=="pre" else "#FF7A6B" for p in ts.period]
ax.bar(range(len(ts)),ts.N,color=col,width=1.0)
ax.axhline(thrN,ls=":",color="#C23A2B",lw=1.5,label=f"임계 평소+3σ={thrN:.0f}")
ax.axvline((ts.bucket<EVENT_TIME).sum()-0.5,color="gray",lw=1)
for _,r in peaks.iterrows():
    xi=int(ts.index[ts.bucket==r.bucket][0])
    ax.annotate(r.bucket.strftime("%y/%m/%d"),(xi,r.N),fontsize=7,rotation=90,va="bottom",ha="center",color="darkred")
st=max(1,len(ts)//24); ax.set_xticks(range(0,len(ts),st))
ax.set_xticklabels([ts.bucket.iloc[i].strftime("%y-%m-%d") for i in range(0,len(ts),st)],rotation=45,ha="right",fontsize=8)
ax.set_ylabel("기사 수 N"); ax.legend(fontsize=9)
ax.set_title(f"{TOPIC_NAME} — 임계 초과(급증) 시점 = 중대 이슈 발생 시점")
plt.tight_layout(); plt.savefig(PNG_SIGNAL,dpi=120); plt.show()

## [8] 언론사 간 유사도
이슈 기간(사후)에 기사가 많은 상위 언론사들이 **서로 얼마나 닮은 기사를 쓰는지**를 봅니다. 언론사별 기사 평균벡터(centroid)를 구해 **언론사×언론사 코사인 행렬**을 만들고 히트맵으로 표시합니다. 값이 높을수록 두 매체가 같은 프레임·문장을 공유(동조)한다는 뜻입니다.

In [ ]:
# ================================================================
# [8] 언론사 간 유사도 — centroid 기반 언론사×언론사 코사인 히트맵
# ================================================================

K=15                                  # 상위 언론사 수
seg = df[df.published_at>=EVENT_TIME]  # 이슈 기간(사후). 전체로 보려면 seg=df
top = seg["press"].value_counts().head(K)
names = list(top.index)
# 언론사별 centroid(평균벡터) → 재정규화
C=[]
for p in names:
    v=emb[seg.index[seg.press==p].values]
    c=v.mean(0); C.append(c/ (np.linalg.norm(c)+1e-9))
C=np.vstack(C); M=C@C.T                # 언론사×언론사 코사인
np.fill_diagonal(M,np.nan)

# 각 언론사의 '동조도' = 다른 매체와의 평균 코사인
sync = pd.Series(np.nanmean(M,axis=1), index=names).sort_values(ascending=False)
print("[언론사별 타 매체와의 평균 유사도(동조도) 상위]")
for nm,val in sync.head(K).items(): print(f"  {val:.3f}  {nm}  (기사 {int(top[nm])})")

fig,ax=plt.subplots(figsize=(10,8))
im=ax.imshow(np.nan_to_num(M,nan=1.0),cmap="YlOrRd",vmin=np.nanmin(M),vmax=np.nanmax(M))
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(names,rotation=90,fontsize=8); ax.set_yticklabels(names,fontsize=8)
for i in range(K):
    for j in range(K):
        if i!=j: ax.text(j,i,f"{M[i,j]:.2f}",ha="center",va="center",fontsize=6,color="black")
fig.colorbar(im,fraction=0.046,pad=0.04,label="코사인 유사도")
ax.set_title(f"{TOPIC_NAME} — 이슈 기간 언론사 간 기사 유사도 (상위 {K}개사)")
plt.tight_layout(); plt.savefig(PNG_MEDIA,dpi=120); plt.show()
print("\n→", PNG_MEDIA, "저장")